In [426]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

[1] "Connected to : yhcr-prd-bradfor-bia-core"


#### Read in data and collect tables

In [427]:
person_ids <- read.csv('data/person_ids.csv', header = TRUE)

In [428]:
school_data = "CB_2489.cb_InstitutionHistory"

In [429]:
school_match = "CB_2489.cb_School"

In [430]:
school_link_data = "CB_2489.cb_InstitutionLink"

In [431]:
enrol_data = "CB_2489.cb_Enrollment"

In [432]:
school_match_table <- tbl(con, school_match) |>
    select(SchoolID, URN
           ) 

In [433]:
enrol_table <- tbl(con, enrol_data) |>
    select(person_id, SchoolID, AcademicYear, 
           OnRoll_1, OnRoll_2, OnRoll_3, 
           EnrolStatus, EntryDate, LeavingDate, PartTime
           ) 

In [434]:
school_table <- tbl(con, school_data) |>
    select(InstitutionHistoryID, InstitutionID, AcademicYear,
           InstitutionTypeDesc, URN, SchoolName,
           PhaseOfEducationDesc
           ) 

In [435]:
school_link_table <- tbl(con, school_link_data) |>
    select(InstitutionLinkID, InstitutionHistoryID, AcademicYear,
           person_id
           ) 

In [436]:
school_match_df <- collect(school_match_table)

In [437]:
head(school_match_df)

SchoolID,URN
<dbl>,<dbl>
3978,108234
1935,108274
21112,108314
11201,108189
14610,139568
3644,108238


In [438]:
school_match <- school_match_df |>
    filter(SchoolID %in% c(cohort_schools)) 

In [439]:
head(school_match)

SchoolID,URN
<dbl>,<dbl>
1935,108274
749,108275
724,108297
2611,108299
2238,108279
11068,135961


In [440]:
enrol_df <- collect(enrol_table)

In [441]:
school_df <- collect(school_table)

In [442]:
school_link_df <- collect(school_link_table)

In [443]:
school_link_filtered <- school_link_df |>
    filter(person_id %in% person_ids$person_id) 

In [444]:
school_link_filtered

InstitutionLinkID,InstitutionHistoryID,AcademicYear,person_id
<dbl>,<dbl>,<chr>,<chr>


#### Filter enrolments table to target cohorts  

In [445]:
enrol_filtered <- enrol_df |>
    filter(person_id %in% person_ids$person_id) 

In [446]:
enrol_filtered |> arrange(AcademicYear) |> distinct(AcademicYear) |> pull(AcademicYear)

[1] "2006/2007" "2007/2008" "2008/2009" "2009/2010" "2010/2011" "2011/2012"
 [7] "2012/2013" "2013/2014" "2014/2015" "2015/2016" "2016/2017" "2017/2018"
[13] "2018/2019"

In [447]:
enrol_filtered <- enrol_filtered |>
    left_join(person_ids, by = join_by(person_id))

In [448]:
enrol_filtered <- enrol_filtered |>
    filter(NCCIS_ACADYR == '2017/2018' & AcademicYear %in% c('2012/2013','2013/2014','2014/2015','2015/2016','2016/2017') | 
           NCCIS_ACADYR == '2018/2019' & AcademicYear %in% c('2013/2014','2014/2015','2015/2016','2016/2017', '2017/2018') 
           )

In [449]:
enrol_df |> 
    filter(person_id =='0D7E6767D93505F4E2D275095D8E4E341CF3F64D21181C133D076B48E194C28A')

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>
0D7E6767D93505F4E2D275095D8E4E341CF3F64D21181C133D076B48E194C28A,6152,2011/2012,1,1,0,C,2008-09-26,NA,0
0D7E6767D93505F4E2D275095D8E4E341CF3F64D21181C133D076B48E194C28A,6152,2008/2009,1,1,1,C,2008-09-26,NA,0
0D7E6767D93505F4E2D275095D8E4E341CF3F64D21181C133D076B48E194C28A,7877,2006/2007,1,0,0,C,2004-01-05,2007-02-23,0
0D7E6767D93505F4E2D275095D8E4E341CF3F64D21181C133D076B48E194C28A,6152,2010/2011,1,1,1,C,2008-09-26,NA,0
0D7E6767D93505F4E2D275095D8E4E341CF3F64D21181C133D076B48E194C28A,6152,2009/2010,1,1,1,C,2008-09-26,NA,0


In [450]:
enrol_filtered |>
    filter(SchoolID == 1)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>
FE3895C54AD1AFAF3487E0DC4AE5D797D949AD1A6BFD767C2A250826F71E8299,1,2013/2014,NA,1,NA,C,2014-03-24,NA,0,2017/2018
865CD8F695ABA1CD8C7DC9652A565843C6522AD06E83B3E687037D6F31DBE3CB,1,2013/2014,NA,0,NA,NA,2014-03-06,2014-04-04,0,2018/2019
4458A610F43FDE3869DF782E3CDF5D5BBD87B1440B3B034BDD3C9325CF95D2D7,1,2013/2014,1,1,NA,C,2013-09-02,NA,0,2018/2019
C61E2B9BB83AC117227A79E3A023D60377E26D488D9B516886AA8584F2315D10,1,2013/2014,0,NA,NA,C,2013-09-02,2013-12-05,0,2018/2019
48A91215B096D39A8D6903838735D2B6F1E9A592794B102E69C81B4157C95CDD,1,2013/2014,1,1,NA,C,2013-11-07,NA,0,2017/2018
E4FEFEA909D85AEB5E03E6E258ED980241087F9B0EAB9BE87065F6DCED31DC72,1,2013/2014,1,1,NA,C,2013-09-02,NA,0,2018/2019
702FB1F91C19B856BFCF9AA812155786E1E44A5873EB0CBEE7D9E789250F31DD,1,2013/2014,1,1,NA,C,2013-11-11,NA,0,2017/2018
428682AC0C308690F8194998ACED59DCC66844610B4CD8E878A66B769015116D,1,2013/2014,1,1,NA,C,2013-09-02,NA,0,2018/2019
551B619FEC7FC59F080F7D0607E4CF9A72E1D194BBDEE479DE181E5BB7F57F0F,1,2013/2014,0,NA,NA,C,2013-09-02,2013-12-05,0,2018/2019


In [451]:
enrol_filtered |> arrange(person_id, AcademicYear)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,980,2012/2013,1,1,1,C,2012-09-04,NA,0,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,980,2013/2014,1,1,1,C,2012-09-04,NA,0,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,980,2014/2015,1,1,1,C,2012-09-04,NA,0,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,980,2015/2016,1,1,NA,C,2012-09-04,NA,0,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,17485,2016/2017,1,1,0,C,2016-10-01,2017-06-30,0,2017/2018
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2204,2012/2013,1,1,1,C,2012-09-04,NA,0,2017/2018
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2204,2013/2014,1,1,NA,C,2012-09-04,NA,0,2017/2018
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,16026,2014/2015,1,1,1,C,2014-09-01,NA,0,2017/2018
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,16026,2015/2016,1,1,1,C,2014-09-01,NA,0,2017/2018


#### Get list of school IDs in enrollments table

In [452]:
cohort_schools <- enrol_filtered |> arrange(SchoolID) |> distinct(SchoolID) |> pull(SchoolID)

In [453]:
length(cohort_schools)

[1] 2053

In [454]:
head(school_match)

SchoolID,URN
<dbl>,<dbl>
1935,108274
749,108275
724,108297
2611,108299
2238,108279
11068,135961


#### Join schools info into school match IDs

In [455]:
head(school_df)

InstitutionHistoryID,InstitutionID,AcademicYear,InstitutionTypeDesc,URN,SchoolName,PhaseOfEducationDesc
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
968,3198,2005/2006,Community school (CY),100222,Craven Park School,Primary
25892,8275,2005/2006,Community school (CY),105771,Hill Top Community Primary School,Primary
40609,11287,2005/2006,Community school (CY),109069,Radstock Infant School,Primary
56451,14508,2005/2006,Independent schools (other) (IND),112448,Holme Park School,Not applicable
123604,28104,2005/2006,Community school (CY),130095,Wembley Primary School,Primary
124903,28374,2005/2006,Further Educational sector college,130541,"Park Lane College, Leeds",16 Plus


In [456]:
school_df$URN <- as.numeric(school_df$URN)

In [457]:
schools_filtered <- school_match |>
    left_join(school_df, by = join_by(URN)) |>
    select(SchoolID, URN, AcademicYear, InstitutionTypeDesc, SchoolName, PhaseOfEducationDesc)

In [458]:
schools_filtered <- schools_filtered |>
    distinct()

In [459]:
head(schools_filtered)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
1935,108274,2005/2006,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2006/2007,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2007/2008,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2008/2009,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2009/2010,Community school (CY),Hemsworth Arts and Community College,Secondary
1935,108274,2010/2011,Community School (CY),Hemsworth Arts and Community College,Secondary


In [460]:
schools_filtered |>
    filter(SchoolID == 6152)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
6152,107211,2005/2006,Community school (CY),Lapage Primary School and Nursery,Primary
6152,107211,2006/2007,Community school (CY),Lapage Primary School and Nursery,Primary
6152,107211,2007/2008,Community school (CY),Lapage Primary School and Nursery,Primary
6152,107211,2008/2009,Community school (CY),Lapage Primary School and Nursery,Primary
6152,107211,2009/2010,Community school (CY),Lapage Primary School and Nursery,Primary
6152,107211,2010/2011,Community School (CY),Lapage Primary School and Nursery,Primary
6152,107211,2011/2012,Community School (CY),Lapage Primary School and Nursery,Primary
6152,107211,2012/2013,Community School (CY),Lapage Primary School and Nursery,Primary
6152,107211,2013/2014,Community School (CY),Lapage Primary School and Nursery,Primary


#### Clean up schools data

In [461]:
# fill NAs in the Phase of Edu col
schools_filtered <- schools_filtered |>
  group_by(URN) |>
  mutate(
    PhaseOfEducationDesc = if_else(
      is.na(PhaseOfEducationDesc),
      first(na.omit(PhaseOfEducationDesc)),
      PhaseOfEducationDesc
    )
  ) |>
  ungroup()

In [462]:
# keep only first row per academic year where multiple per school (e.g. change in school type mid year to academy status)
schools_filtered <- schools_filtered |>
  arrange(AcademicYear) |>
  group_by(SchoolID, URN, AcademicYear) |>
  slice(1) |>
  ungroup()

In [463]:
schools_filtered |> 
    filter(SchoolID == 21)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
21,100747,2005/2006,Community school (CY),Crofton School,Secondary
21,100747,2006/2007,Community school (CY),Crofton School,Secondary
21,100747,2007/2008,Community school (CY),Crofton School,Secondary
21,100747,2008/2009,Community school (CY),Prendergast - Ladywell Fields College,Secondary
21,100747,2009/2010,Community school (CY),Prendergast - Ladywell Fields College,Secondary
21,100747,2010/2011,Community School (CY),Prendergast - Ladywell Fields College,Secondary
21,100747,2011/2012,Community School (CY),Prendergast - Ladywell Fields College,Secondary
21,100747,2012/2013,Community School (CY),Prendergast - Ladywell Fields College,Secondary
21,100747,2013/2014,Community School (CY),Prendergast - Ladywell Fields College,Secondary


In [464]:
# Check if SchoolID + AcademicYear is unique 
schools_filtered |>
  count(SchoolID, AcademicYear) |>
  filter(n > 1)  # problem rows

SchoolID,AcademicYear,n
<dbl>,<chr>,<int>


#### clean up institution types

In [465]:
schools_filtered |>
    distinct(InstitutionTypeDesc)

InstitutionTypeDesc
<chr>
Community school (CY)
Community School (CY)
Voluntary aided school (VA)
Voluntary Aided School (VA)
Foundation School (FD)
Voluntary controlled school (VC)
Voluntary Controlled School (VC)
Foundation school (FD)
Academy (AC)


In [466]:
schools_filtered <- schools_filtered |>
  mutate(InstitutionTypeDesc = str_replace_all(InstitutionTypeDesc, "û", "-"))

In [467]:
schools_filtered <- schools_filtered |>
  mutate(InstitutionTypeDesc = InstitutionTypeDesc |> 
           str_trim())

In [468]:
schools_filtered <- schools_filtered |>
    mutate(InstitutionTypeDesc = case_when(
            InstitutionTypeDesc %in% c("Community school (CY)", "Community School (CY)") ~ 'Community school',
        
        InstitutionTypeDesc %in% c("Voluntary aided school (VA)", "Voluntary Aided School (VA)") ~ 'Voluntary aided school',
        
        InstitutionTypeDesc %in% c("Foundation School (FD)", "Foundation school (FD)") ~ 'Foundation school',
        
        InstitutionTypeDesc %in% c("Voluntary controlled school (VC)", "Voluntary Controlled School (VC)") ~ 'Voluntary controlled school',
        
        InstitutionTypeDesc %in% c("Academy (AC)", "Academy Schhol", "Academy Free Schools / Consortia") ~ 'Academy',
        
        InstitutionTypeDesc %in% c("Academy Sponsor Led (AC)", "Academy - Sponsor Led Mainstream (AC)") ~ 'Academy sponsor led',
        
        InstitutionTypeDesc %in% c("City technology college (CTC)", "City technology College (CTC)", "City Technology College (CTC)") ~ 'City technology college',
        
        InstitutionTypeDesc %in% c("Community special school (CYS)", "Community Special School (CYS)") ~ 'Community special school',
        
        InstitutionTypeDesc %in% c("Special school not maintained by LEA (NMSS)", "Non-Maintained Special School (NMSS)") ~ 'Non-maintained special school',
        
        InstitutionTypeDesc %in% c("Foundation special school (FDS)", "Foundation Special School (FDS)") ~ 'Foundation special school',
        
        InstitutionTypeDesc %in% c("Community hospital school (CYH)") ~ 'Community hospital school',
        
        InstitutionTypeDesc %in% c("Academy Converters Mainstream", "Academy Converter - Mainstream (ACC)") ~ 'Academy converter',
        
        InstitutionTypeDesc %in% c("Free School - Mainstream (F)", "Community School (CY)") ~ 'Free school',
        
        InstitutionTypeDesc %in% c("Academy - Converter Special School (ACCS)", "Academy - Sponsor Led Special School (ACS)") ~ 'Academy special school',
        
        InstitutionTypeDesc %in% c("Pupil referral unit", "Pupil Referral Unit", 
                                   'Pupil Referral Unit (PRU)', 'Academy - Converter AP (ACCAP)',
                                  'Free School - Alternative Provision', 'Academy - Sponsor Led AP (ACAP)',
                                  'Free School - Alternative Provision (FAP)') ~ 'AP/PRU',
        
        InstitutionTypeDesc %in% c("Free School - Studio School (FSS)", 'Free School - Studio School') ~ 'Studio school',
        InstitutionTypeDesc %in% c("Free School - UTC (FUTC)") ~ 'University technical college',
        InstitutionTypeDesc %in% c("Free School - Studio School (FSS)") ~ 'Studio school',
        InstitutionTypeDesc %in% c("Free School - 16-19 (F1619)") ~ 'Free school (16-19)',
    TRUE ~ InstitutionTypeDesc)
           )
        
        

#### check for schools changing name

In [469]:
schools_filtered <- schools_filtered |>
  mutate(SchoolName = gsub("[[:punct:]]", "", SchoolName))

In [470]:
schools_filtered <- schools_filtered |>
  mutate(SchoolName = SchoolName |> 
           str_trim())

In [471]:
schools_filtered |>
    group_by(URN) |> 
    filter(n_distinct(SchoolName) > 1) |> 
    arrange(URN, SchoolName)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
1706,100051,2012/2013,Community school,Regent High School,Secondary
1706,100051,2013/2014,Community school,Regent High School,Secondary
1706,100051,2014/2015,Community school,Regent High School,Secondary
1706,100051,2015/2016,Community school,Regent High School,Secondary
1706,100051,2016/2017,Community school,Regent High School,Secondary
1706,100051,2017/2018,Community school,Regent High School,Secondary
1706,100051,2018/2019,Community school,Regent High School,Secondary
1706,100051,2019/2020,Community school,Regent High School,Secondary
1706,100051,2020/2021,Community school,Regent High School,Secondary


In [472]:
latest_names <- schools_filtered |>
    group_by(URN) |>
    slice_max(AcademicYear, n = 1, with_ties = FALSE) |>
    select(URN, SchoolName) |>
    rename(LatestSchoolName = SchoolName)

In [473]:
head(latest_names)

URN,LatestSchoolName
<dbl>,<chr>
100051,Regent High School
100192,The John Roan School
100277,Haggerston School
100279,Stoke Newington School and Sixth Form
100284,The Urswick School A Church of England Secondary School
100360,Fulham Cross Girls School and Language College


In [474]:
latest_names |>
    filter(URN == 107302)

URN,LatestSchoolName
<dbl>,<chr>
107302,All Saints CofE Primary School


In [475]:
schools_filtered <- schools_filtered |>
    left_join(latest_names, by = "URN") |>
    mutate(SchoolName = coalesce(LatestSchoolName, SchoolName)) |>
    select(-LatestSchoolName)

In [476]:
head(schools_filtered)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
1,108056,2005/2006,Community school,City of Leeds School,Secondary
1,108056,2006/2007,Community school,City of Leeds School,Secondary
1,108056,2007/2008,Community school,City of Leeds School,Secondary
1,108056,2008/2009,Community school,City of Leeds School,Secondary
1,108056,2009/2010,Community school,City of Leeds School,Secondary
1,108056,2010/2011,Community school,City of Leeds School,Secondary


#### fuzzy match for slightly different name spellings

In [243]:
install.packages('stringdist')

Installing package into ‘/home/jupyter/.R/library’
(as ‘lib’ is unspecified)



In [244]:
library(stringdist)

In [284]:
school_names <- unique(schools_filtered$SchoolName)

In [285]:
dist_matrix <- stringdistmatrix(school_names, school_names, method = "jw")

In [286]:
# Find likely duplicates based on low distance
fuzzy_matches <- which(dist_matrix < 0.1 & dist_matrix > 0, arr.ind = TRUE)

In [301]:
enrol_final |>
    filter(SchoolName == 'Queen Elizabeths School')

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc,LatestURN
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>
BB1C7F4FC50A60FF728A48C7F2A3C4496F2E6765FA62119EBBF5ABB6FB133735,16226,2015/2016,1,1,1,C,2015-09-01,NA,0,2018/2019,141526,Academy converter,Queen Elizabeths School,Secondary,141526
BB1C7F4FC50A60FF728A48C7F2A3C4496F2E6765FA62119EBBF5ABB6FB133735,16226,2016/2017,1,1,1,C,2015-09-01,NA,0,2018/2019,141526,Academy converter,Queen Elizabeths School,Secondary,141526
BB1C7F4FC50A60FF728A48C7F2A3C4496F2E6765FA62119EBBF5ABB6FB133735,16226,2017/2018,1,1,1,C,2015-09-01,NA,0,2018/2019,141526,Academy converter,Queen Elizabeths School,Secondary,141526
D5BC71328A6A3D4C7F394FF461AB2209C88B7569BF0CFFDA0A73A87D0AB8607C,16226,2015/2016,1,0,NA,C,2015-09-02,2016-02-26,0,2018/2019,141526,Academy converter,Queen Elizabeths School,Secondary,141526


In [303]:
enrol_final |>
    filter(SchoolName == "Queen Elizabeths Grammar School")

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc,LatestURN
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>
5E067A183AF6A1DFCA5B9A38548CB42F4131EE05E131A525F083F605CCFADB55,16035,2017/2018,1,1,1,C,2014-09-22,NA,0,2018/2019,141165,Free school,Queen Elizabeths Grammar School,NA,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2013/2014,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2016/2017,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
5E067A183AF6A1DFCA5B9A38548CB42F4131EE05E131A525F083F605CCFADB55,16035,2014/2015,1,1,1,C,2014-09-22,NA,0,2018/2019,141165,Free school,Queen Elizabeths Grammar School,NA,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2014/2015,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2015/2016,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
5E067A183AF6A1DFCA5B9A38548CB42F4131EE05E131A525F083F605CCFADB55,16035,2015/2016,1,1,1,C,2014-09-22,NA,0,2018/2019,141165,Free school,Queen Elizabeths Grammar School,NA,136972
8F2BE8D2C898930C38AA9528327C1F76996160EED76CE10B90E666D588D3E436,11607,2012/2013,1,1,1,C,2012-09-03,NA,0,2017/2018,136972,Academy converter,Queen Elizabeths Grammar School,Secondary,136972
5E067A183AF6A1DFCA5B9A38548CB42F4131EE05E131A525F083F605CCFADB55,16035,2016/2017,1,1,1,C,2014-09-22,NA,0,2018/2019,141165,Free school,Queen Elizabeths Grammar School,NA,136972


In [287]:
tibble(
  name1 = school_names[fuzzy_matches[, 1]],
  name2 = school_names[fuzzy_matches[, 2]],
  distance = dist_matrix[fuzzy_matches]
) %>%
  distinct() %>%
  arrange(distance)

name1,name2,distance
<chr>,<chr>,<dbl>
Queen Elizabeths School,Queen Elizabeth School,0.01449275
Queen Elizabeth School,Queen Elizabeths School,0.01449275
Hinde House 216 School,Hinde House 316 School,0.03030303
Hinde House 316 School,Hinde House 216 School,0.03030303
Bedale High School,Beal High School,0.03703704
Forest Hall School,Forest Hill School,0.03703704
Beal High School,Bedale High School,0.03703704
Forest Hill School,Forest Hall School,0.03703704
Brookfield School,Broomfield School,0.03921569


In [ ]:
#school_names_clean <- schools_filtered |>
#    mutate(SchoolName = case_when(
#            SchoolName %in% c("Hinde House 216 School", "Hinde House 316 School") ~ 'Hinde House 2-16 School',
#        SchoolName %in% c("St Josephs Catholic College", "St Bedes and St Josephs Catholic College") ~ 'St Bedes and St Josephs Catholic College',
#        SchoolName %in% c("Oakbank School", "Beckfoot Oakbank") ~ 'Beckfoot Oakbank',
#        SchoolName %in% c("Tong High School", "Tong Leadership Academy") ~ 'Tong Leadership Academy',
#        SchoolName %in% c("St Anselms Catholic School", "St Anselms Catholic School Canterbury") ~ 'Hinde House 2-16 School',

In [307]:
enrol_final |>
    filter(SchoolName == "St Anselms Catholic School")

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc,LatestURN
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>
329745E3565E94BD632FA7AB0F347C5E1C5B5DDE0E32304B542436108D440D9F,1505,2012/2013,NA,1,1,C,2013-05-07,NA,0,2017/2018,118918,Voluntary aided school,St Anselms Catholic School,Secondary,118918
329745E3565E94BD632FA7AB0F347C5E1C5B5DDE0E32304B542436108D440D9F,1505,2013/2014,1,1,1,M,2013-05-07,NA,0,2017/2018,118918,Voluntary aided school,St Anselms Catholic School,Secondary,118918
296E09A08A59BA0C6C6491466160F8E03009E1F05A3F75C0ADA1D848F541F631,1505,2013/2014,1,1,1,C,2013-09-03,NA,0,2018/2019,118918,Voluntary aided school,St Anselms Catholic School,Secondary,118918


#### Decision - treat academisation as new school - DfE do as asign new URN

#### IGNORE check for schools changing URN because of becoming academies etc... 

In [ ]:
#schools_filtered |>
    group_by(SchoolName) |> 
    filter(n_distinct(URN) > 1) |> 
    arrange(SchoolName, URN)

In [289]:
#latest_urn <- schools_filtered |>
    group_by(SchoolName) |>
    slice_max(AcademicYear, n = 1, with_ties = FALSE) |>
    select(URN, SchoolName) |>
    rename(LatestURN = URN)

In [308]:
#head(latest_urn)

In [291]:
#schools_filtered <- schools_filtered |>
  left_join(latest_urn, by = "SchoolName")

In [329]:
schools_filtered |>
    group_by(SchoolName) |> 
    filter(n_distinct(URN) > 1) |> 
    arrange(SchoolName, URN)

SchoolID,URN,AcademicYear,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
10414,135622,2007/2008,Academy,Academy 360,Not applicable
10414,135622,2008/2009,Academy,Academy 360,Not applicable
10414,135622,2009/2010,Academy,Academy 360,Not applicable
10414,135622,2010/2011,Academy sponsor led,Academy 360,Not applicable
10414,135622,2011/2012,Academy sponsor led,Academy 360,Not applicable
10414,135622,2012/2013,Academy sponsor led,Academy 360,Not applicable
10414,135622,2013/2014,Academy sponsor led,Academy 360,Secondary
10414,135622,2014/2015,Academy sponsor led,Academy 360,Not applicable
10414,135622,2015/2016,Academy sponsor led,Academy 360,Not applicable


#### Save school info to csv

In [477]:
write.csv(schools_filtered, "data/schools_filtered.csv", row.names = FALSE)

#### Join matched schools data to enrolments table

In [480]:
enrol_final <- enrol_filtered |>
    left_join(schools_filtered, by = join_by(SchoolID, AcademicYear))

In [481]:
enrol_final |>
    arrange(person_id, AcademicYear)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,980,2012/2013,1,1,1,C,2012-09-04,NA,0,2017/2018,107442,Foundation school,Thornton Grammar School,Secondary
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,980,2013/2014,1,1,1,C,2012-09-04,NA,0,2017/2018,107442,Foundation school,Thornton Grammar School,Secondary
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,980,2014/2015,1,1,1,C,2012-09-04,NA,0,2017/2018,107442,Foundation school,Thornton Grammar School,Secondary
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,980,2015/2016,1,1,NA,C,2012-09-04,NA,0,2017/2018,107442,Foundation school,Thornton Grammar School,Secondary
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,17485,2016/2017,1,1,0,C,2016-10-01,2017-06-30,0,2017/2018,143114,Academy sponsor led,Beckfoot Thornton,Secondary
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2204,2012/2013,1,1,1,C,2012-09-04,NA,0,2017/2018,107429,Voluntary aided school,St Josephs Catholic College,Secondary
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2204,2013/2014,1,1,NA,C,2012-09-04,NA,0,2017/2018,107429,Voluntary aided school,St Josephs Catholic College,Secondary
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,16026,2014/2015,1,1,1,C,2014-09-01,NA,0,2017/2018,140569,Voluntary aided school,St Bedes and St Josephs Catholic College,Secondary
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,16026,2015/2016,1,1,1,C,2014-09-01,NA,0,2017/2018,140569,Voluntary aided school,St Bedes and St Josephs Catholic College,Secondary


#### check how many school moves in the data

In [482]:
id_lists <- enrol_final |>
    select(person_id, SchoolName)

In [483]:
#id_lists

In [484]:
# n = 85,423
head(id_lists)

person_id,SchoolName
<chr>,<chr>
F65B08C0ABA0DF84D9BD65F7FED88E1D5DAD42AAB47392CE2D9543AED4E245DD,South Craven School
F97D233CF64E3EAC93369EAAA28D9E5193785F4BA369A53A716A1B453B9FDDA2,Appleton Academy
3E4432F9D7005D74AB24D68A875EC38A8DA0234A2D67666F935999C059682696,Gloucester Academy
823AECF1EFC7F8CD963B2B16D2C886922CD9F7C0CA4B478BD07E86F163685822,Endeavour High School
18746FF26D6AC95D757A4B4CB10B01DE6F041115B43ECC8BE0C4A0AEF575224B,Coop Academy Leeds
0D5BABE74A35CC1F64DB2C1F97898AD55B2DF12975FE992FE28487B958392B33,Appleton Academy


In [485]:
distinct(id_lists)

person_id,SchoolName
<chr>,<chr>
F65B08C0ABA0DF84D9BD65F7FED88E1D5DAD42AAB47392CE2D9543AED4E245DD,South Craven School
F97D233CF64E3EAC93369EAAA28D9E5193785F4BA369A53A716A1B453B9FDDA2,Appleton Academy
3E4432F9D7005D74AB24D68A875EC38A8DA0234A2D67666F935999C059682696,Gloucester Academy
823AECF1EFC7F8CD963B2B16D2C886922CD9F7C0CA4B478BD07E86F163685822,Endeavour High School
18746FF26D6AC95D757A4B4CB10B01DE6F041115B43ECC8BE0C4A0AEF575224B,Coop Academy Leeds
0D5BABE74A35CC1F64DB2C1F97898AD55B2DF12975FE992FE28487B958392B33,Appleton Academy
A73E48D704D747B6822F8D7DAE261C7F719BB3B878CDB3FD1973B4C3567F41B7,Appleton Academy
8D925CADDE220244727D160F216F678F9D0871E5857020A5DBBAFA272F6F043C,Appleton Academy
E4E0CCAD6B73017977DB911E9788BB7A4147275CF81C7872FB2636976920FB14,Appleton Academy


In [486]:
# n = 24,688
id_lists_distinct <- distinct(id_lists)

In [487]:
person_counts <- table(id_lists_distinct$person_id)
summary <- table(person_counts)

In [488]:
as.data.frame(summary)

person_counts,Freq
<fct>,<int>
1,10792
2,5050
3,808
4,219
5,70
6,18
7,2
8,3


#### Explore high school count ids

In [489]:
enrol_final |> 
    filter(person_id == '38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F') |>
    arrange(AcademicYear, EntryDate)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,1562,2012/2013,1,1,1,C,2012-09-03,NA,0,2017/2018,107440,Foundation school,Hanson School,Secondary
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,1562,2013/2014,1,0,0,C,2012-09-03,2014-04-03,0,2017/2018,107440,Foundation school,Hanson School,Secondary
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,15223,2013/2014,0,NA,NA,NA,2013-09-24,2013-12-13,0,2017/2018,133411,AP/PRU,Bradford Central PRU,Not applicable
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,344,2013/2014,NA,1,1,C,2014-04-04,NA,0,2017/2018,107563,Community school,Sowerby Bridge High School,Secondary
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,344,2014/2015,1,0,0,M,2014-04-04,2015-02-23,0,2017/2018,107563,Community school,Sowerby Bridge High School,Secondary
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,14447,2014/2015,0,NA,NA,S,2014-10-02,2014-11-14,0,2017/2018,133693,AP/PRU,Calderdale PRU,Not applicable
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,14447,2015/2016,1,1,1,C,2014-10-02,NA,0,2017/2018,133693,AP/PRU,Calderdale PRU,Not applicable
38C38B72800AB59D24A9677F19DB0C3872F37209D24BC03DA9F68E7543CF333F,14447,2016/2017,1,1,0,C,2014-10-02,2017-06-30,0,2017/2018,133693,AP/PRU,Calderdale PRU,Not applicable


#### create a table of person_id and school count

In [490]:
# create a new col with count to match with destinations
school_count <- id_lists_distinct %>%
  group_by(person_id) %>%
  mutate(n_schools = n()) %>%
  ungroup()

In [491]:
head(school_count)

person_id,SchoolName,n_schools
<chr>,<chr>,<int>
F65B08C0ABA0DF84D9BD65F7FED88E1D5DAD42AAB47392CE2D9543AED4E245DD,South Craven School,2
F97D233CF64E3EAC93369EAAA28D9E5193785F4BA369A53A716A1B453B9FDDA2,Appleton Academy,3
3E4432F9D7005D74AB24D68A875EC38A8DA0234A2D67666F935999C059682696,Gloucester Academy,2
823AECF1EFC7F8CD963B2B16D2C886922CD9F7C0CA4B478BD07E86F163685822,Endeavour High School,2
18746FF26D6AC95D757A4B4CB10B01DE6F041115B43ECC8BE0C4A0AEF575224B,Coop Academy Leeds,1
0D5BABE74A35CC1F64DB2C1F97898AD55B2DF12975FE992FE28487B958392B33,Appleton Academy,2


# Pivot Long for terms then Pivot Wide for years

In [492]:
head(enrol_final)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
F65B08C0ABA0DF84D9BD65F7FED88E1D5DAD42AAB47392CE2D9543AED4E245DD,12433,2017/2018,1,1,1,C,NA,NA,0,2018/2019,136736,Academy converter,South Craven School,Secondary
F97D233CF64E3EAC93369EAAA28D9E5193785F4BA369A53A716A1B453B9FDDA2,10875,2014/2015,0,0,NA,NA,2014-11-10,2015-01-08,0,2017/2018,135865,Academy sponsor led,Appleton Academy,Not applicable
3E4432F9D7005D74AB24D68A875EC38A8DA0234A2D67666F935999C059682696,11365,2017/2018,1,1,0,C,2014-11-10,2018-06-29,0,2018/2019,136199,Academy sponsor led,Gloucester Academy,Secondary
823AECF1EFC7F8CD963B2B16D2C886922CD9F7C0CA4B478BD07E86F163685822,1431,2012/2013,1,1,0,C,2012-10-04,NA,0,2017/2018,133422,Community school,Endeavour High School,Secondary
18746FF26D6AC95D757A4B4CB10B01DE6F041115B43ECC8BE0C4A0AEF575224B,14276,2016/2017,1,1,0,C,2012-10-04,2017-06-30,0,2017/2018,137065,Academy sponsor led,Coop Academy Leeds,Secondary
0D5BABE74A35CC1F64DB2C1F97898AD55B2DF12975FE992FE28487B958392B33,18286,2017/2018,1,1,1,C,2017-09-01,NA,0,2018/2019,145173,Academy sponsor led,Appleton Academy,NA


In [493]:
school_per_term <- enrol_final |>
    select(-c(SchoolID, PartTime, SchoolName, InstitutionTypeDesc, PhaseOfEducationDesc))

In [494]:
head(school_per_term)

person_id,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>
F65B08C0ABA0DF84D9BD65F7FED88E1D5DAD42AAB47392CE2D9543AED4E245DD,2017/2018,1,1,1,C,NA,NA,2018/2019,136736
F97D233CF64E3EAC93369EAAA28D9E5193785F4BA369A53A716A1B453B9FDDA2,2014/2015,0,0,NA,NA,2014-11-10,2015-01-08,2017/2018,135865
3E4432F9D7005D74AB24D68A875EC38A8DA0234A2D67666F935999C059682696,2017/2018,1,1,0,C,2014-11-10,2018-06-29,2018/2019,136199
823AECF1EFC7F8CD963B2B16D2C886922CD9F7C0CA4B478BD07E86F163685822,2012/2013,1,1,0,C,2012-10-04,NA,2017/2018,133422
18746FF26D6AC95D757A4B4CB10B01DE6F041115B43ECC8BE0C4A0AEF575224B,2016/2017,1,1,0,C,2012-10-04,2017-06-30,2017/2018,137065
0D5BABE74A35CC1F64DB2C1F97898AD55B2DF12975FE992FE28487B958392B33,2017/2018,1,1,1,C,2017-09-01,NA,2018/2019,145173


C = Current (single registration at this school)
G = Guest (pupil not registered at this school but attending some lessons or sessions)
M = Current Main (dual registration)
S = Current Subsidiary (dual registration)
F = FE College (since 2014/15)
O = Other provider (since 2014/15)

In [495]:
school_per_term |> distinct(EnrolStatus) |> pull(EnrolStatus)

[1] "C" NA  "S" "O" "M" "F"

In [496]:
school_per_term |> 
    filter(person_id == '56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB' ) |>
    arrange(AcademicYear, EntryDate)

person_id,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2012/2013,0,NA,NA,C,2012-09-03,2012-10-05,2017/2018,107366
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2012/2013,1,1,1,C,2012-10-04,NA,2017/2018,135367
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2013/2014,1,1,1,C,2012-10-04,NA,2017/2018,135367
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2014/2015,1,1,1,M,2012-10-04,NA,2017/2018,135367
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2014/2015,0,NA,NA,S,2014-09-26,2014-11-27,2017/2018,107350
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2014/2015,1,0,NA,C,2014-12-10,2015-02-13,2017/2018,133411
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2015/2016,0,NA,NA,M,2012-10-04,2015-11-06,2017/2018,135367
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2015/2016,1,1,1,O,2015-11-09,NA,2017/2018,135732
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,2016/2017,1,1,0,C,2015-11-09,2017-06-30,2017/2018,135732


In [497]:
enrol_final |> 
    filter(person_id == '56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB' ) |>
    arrange(AcademicYear, EntryDate)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,1716,2012/2013,0,NA,NA,C,2012-09-03,2012-10-05,0,2017/2018,107366,Foundation school,Tong High School,Secondary
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,9574,2012/2013,1,1,1,C,2012-10-04,NA,0,2017/2018,135367,Academy sponsor led,Bradford Academy,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,9574,2013/2014,1,1,1,C,2012-10-04,NA,0,2017/2018,135367,Academy sponsor led,Bradford Academy,Secondary
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,9574,2014/2015,1,1,1,M,2012-10-04,NA,0,2017/2018,135367,Academy sponsor led,Bradford Academy,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,1340,2014/2015,0,NA,NA,S,2014-09-26,2014-11-27,0,2017/2018,107350,Foundation school,Buttershaw Business and Enterprise College,Secondary
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,15223,2014/2015,1,0,NA,C,2014-12-10,2015-02-13,0,2017/2018,133411,AP/PRU,Bradford Central PRU,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,9574,2015/2016,0,NA,NA,M,2012-10-04,2015-11-06,0,2017/2018,135367,Academy sponsor led,Bradford Academy,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,15167,2015/2016,1,1,1,O,2015-11-09,NA,0,2017/2018,135732,AP/PRU,Bradford District PRU,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,15167,2016/2017,1,1,0,C,2015-11-09,2017-06-30,0,2017/2018,135732,AP/PRU,Bradford District PRU,Not applicable


#### pivot long

In [498]:
pivot_long <- school_per_term %>%
  pivot_longer(
    cols = starts_with("OnRoll_"),            
    names_to = "Term",                        # new column name for term
    values_to = "OnRoll",                     # values (0, 1, or NA)
    names_prefix = "OnRoll_"                  # remove this prefix from 'Term'
  )

In [499]:
pivot_long |>
    arrange(person_id, AcademicYear, Term)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012/2013,C,2012-09-04,NA,2017/2018,107442,1,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012/2013,C,2012-09-04,NA,2017/2018,107442,2,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012/2013,C,2012-09-04,NA,2017/2018,107442,3,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013/2014,C,2012-09-04,NA,2017/2018,107442,1,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013/2014,C,2012-09-04,NA,2017/2018,107442,2,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013/2014,C,2012-09-04,NA,2017/2018,107442,3,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2014/2015,C,2012-09-04,NA,2017/2018,107442,1,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2014/2015,C,2012-09-04,NA,2017/2018,107442,2,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2014/2015,C,2012-09-04,NA,2017/2018,107442,3,1


In [500]:
pivot_long |>
  group_by(person_id, AcademicYear, Term) |>
  filter(n() > 1) |>
  arrange(person_id, AcademicYear, Term)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,S,2014-01-07,2014-03-19,2017/2018,133411,1,1
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,C,2012-09-04,NA,2017/2018,107441,1,1
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,S,2014-01-07,2014-03-19,2017/2018,133411,2,0
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,C,2012-09-04,NA,2017/2018,107441,2,1
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,S,2014-01-07,2014-03-19,2017/2018,133411,3,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,C,2012-09-04,NA,2017/2018,107441,3,1
0054192801954D860BB235F844AD1CB8A99956B2CF3D4E20F83C650DD6279B58,2012/2013,NA,2012-09-03,2012-10-02,2017/2018,107440,1,0
0054192801954D860BB235F844AD1CB8A99956B2CF3D4E20F83C650DD6279B58,2012/2013,C,2012-10-03,NA,2017/2018,107413,1,1
0054192801954D860BB235F844AD1CB8A99956B2CF3D4E20F83C650DD6279B58,2012/2013,NA,2012-09-03,2012-10-02,2017/2018,107440,2,NA


In [501]:
pivot_long_clean <- pivot_long |>
    group_by(person_id, AcademicYear, Term) |>
    filter(n() > 1) |>
    mutate(
        # prioritise the record with onroll == T
        onroll_priority = ifelse(OnRoll == "1", 1, 0),
        # prioritise the record with C enrolment, then M enrolment status
        enrol_priority = case_when(
          EnrolStatus == "C" ~ 3,
          EnrolStatus == "M" ~ 2,
          TRUE ~ 1
            ),
        # combine
        priority_score = onroll_priority * 10 + enrol_priority
      ) |>
    # keep single highest priority record
    arrange(desc(priority_score)) |>
    slice_head(n = 1) |>
    ungroup()

In [502]:
non_duplicates <- pivot_long |>
  group_by(person_id, AcademicYear, Term) |>
  filter(n() == 1) |>
  ungroup()

In [503]:
pivot_long_clean <- bind_rows(pivot_long_clean, non_duplicates) |>
  arrange(person_id, AcademicYear, Term)

In [504]:
# n = 246807

In [505]:
pivot_long_clean |>
  #group_by(person_id, AcademicYear, Term) |>
  #filter(n() > 1) |>
  arrange(person_id, AcademicYear, Term)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll,onroll_priority,enrol_priority,priority_score
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012/2013,C,2012-09-04,NA,2017/2018,107442,1,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012/2013,C,2012-09-04,NA,2017/2018,107442,2,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012/2013,C,2012-09-04,NA,2017/2018,107442,3,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013/2014,C,2012-09-04,NA,2017/2018,107442,1,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013/2014,C,2012-09-04,NA,2017/2018,107442,2,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013/2014,C,2012-09-04,NA,2017/2018,107442,3,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2014/2015,C,2012-09-04,NA,2017/2018,107442,1,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2014/2015,C,2012-09-04,NA,2017/2018,107442,2,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2014/2015,C,2012-09-04,NA,2017/2018,107442,3,1,NA,NA,NA


In [506]:
head(pivot_long_clean)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll,onroll_priority,enrol_priority,priority_score
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012/2013,C,2012-09-04,NA,2017/2018,107442,1,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012/2013,C,2012-09-04,NA,2017/2018,107442,2,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012/2013,C,2012-09-04,NA,2017/2018,107442,3,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013/2014,C,2012-09-04,NA,2017/2018,107442,1,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013/2014,C,2012-09-04,NA,2017/2018,107442,2,1,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013/2014,C,2012-09-04,NA,2017/2018,107442,3,1,NA,NA,NA


In [507]:
df_for_pivot_wide <- pivot_long_clean |>
    select(c(person_id, NCCIS_ACADYR, URN, AcademicYear, Term))

In [508]:
head(df_for_pivot_wide)

person_id,NCCIS_ACADYR,URN,AcademicYear,Term
<chr>,<chr>,<dbl>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,107442,2012/2013,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,107442,2012/2013,2
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,107442,2012/2013,3
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,107442,2013/2014,1
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,107442,2013/2014,2
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,107442,2013/2014,3


In [509]:
# Save as csv 
write.csv(df_for_pivot_wide, "data/school_urn_enrollment_perterm.csv", row.names = FALSE)

In [415]:
#enrol_final
pivot_long_clean |> 
    filter(person_id == '004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894' ) |>
    arrange(AcademicYear)

person_id,AcademicYear,EnrolStatus,EntryDate,LeavingDate,NCCIS_ACADYR,URN,Term,OnRoll,onroll_priority,enrol_priority,priority_score
<chr>,<chr>,<chr>,<date>,<date>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2012/2013,C,2012-09-04,NA,2017/2018,107441,1,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2012/2013,C,2012-09-04,NA,2017/2018,107441,2,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2012/2013,C,2012-09-04,NA,2017/2018,107441,3,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,C,2012-09-04,NA,2017/2018,107441,1,1,1,3,13
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,C,2012-09-04,NA,2017/2018,107441,2,1,1,3,13
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2013/2014,C,2012-09-04,NA,2017/2018,107441,3,1,1,3,13
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2014/2015,S,2014-07-07,NA,2017/2018,107428,1,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2014/2015,S,2014-07-07,NA,2017/2018,107428,2,1,NA,NA,NA
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,2014/2015,S,2014-07-07,NA,2017/2018,107428,3,1,NA,NA,NA


#### Pivot Wide

In [418]:
pivot_wide <- df_for_pivot_wide |>
    arrange(person_id, AcademicYear, Term) |>
    mutate(Year_Term = paste0("Y", AcademicYear, "_", "Term", Term),
          URN = as.character(URN)) |>
    select(person_id, NCCIS_ACADYR, Year_Term, URN) |>
    pivot_wider(
        names_from = Year_Term,
        values_from = URN,
        values_fill = NA
      )

In [419]:
pivot_wide

person_id,NCCIS_ACADYR,Y2012/2013_Term1,Y2012/2013_Term2,Y2012/2013_Term3,Y2013/2014_Term1,Y2013/2014_Term2,Y2013/2014_Term3,Y2014/2015_Term1,Y2014/2015_Term2,Y2014/2015_Term3,Y2015/2016_Term1,Y2015/2016_Term2,Y2015/2016_Term3,Y2016/2017_Term1,Y2016/2017_Term2,Y2016/2017_Term3,Y2017/2018_Term1,Y2017/2018_Term2,Y2017/2018_Term3
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,107442,107442,107442,107442,107442,107442,107442,107442,107442,107442,107442,107442,143114,143114,143114,NA,NA,NA
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2017/2018,107429,107429,107429,107429,107429,107429,140569,140569,140569,140569,140569,140569,140569,140569,140569,NA,NA,NA
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018/2019,NA,NA,NA,136529,136529,136529,136529,136529,136529,136529,136529,136529,136529,136529,136529,136529,136529,136529
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2018/2019,NA,NA,NA,107441,107441,107441,107441,107441,107441,107441,107441,107441,143112,143112,143112,143112,143112,143112
00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,2017/2018,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,108087,NA,NA,NA
0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,2017/2018,107366,107366,107366,107366,107366,107366,107366,107366,107366,107366,107366,107366,142761,142761,142761,NA,NA,NA
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,2018/2019,NA,NA,NA,140111,140111,140111,139474,139474,139474,139474,139474,139474,139474,139474,139474,139474,139474,139474
0019031CD62586D59268772800516ED50CDF1EF7DE46488542BB031850F6E68A,2018/2019,NA,NA,NA,139982,139982,139982,139982,139982,139982,139982,139982,139982,139982,139982,139982,139982,139982,139982
001B7CEB7B4CC529920A0B9F5B83D1AC0019E5DC44B3FBB3CEF665C331C0C3B6,2017/2018,107395,107395,107395,107395,107395,107395,107395,107395,107395,107395,107395,107395,107395,107395,107395,NA,NA,NA


#### Save as CSV

In [351]:
write.csv(pivot_long, "data/enrolment_by_term.csv", row.names = FALSE)

In [359]:
write.csv(pivot_long_clean, "data/enrolment_by_term_clean.csv", row.names = FALSE)

In [420]:
write.csv(pivot_wide, "data/enrolment_wide.csv", row.names = FALSE)

In [352]:
write.csv(enrol_final, "data/enrol_final.csv", row.names = FALSE)

In [353]:
write.csv(school_count, "data/school_count.csv", row.names = FALSE)

# extract from full enrolment tables whether ever attended AP/PRU

In [393]:
head(enrol_final)

person_id,SchoolID,AcademicYear,OnRoll_1,OnRoll_2,OnRoll_3,EnrolStatus,EntryDate,LeavingDate,PartTime,NCCIS_ACADYR,URN,InstitutionTypeDesc,SchoolName,PhaseOfEducationDesc
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
F48A1CAEA9F5279772EA10D2BAA19242ABDDEFDE73A02C2EDD1047022BA8C2F6,15177,2016/2017,1,1,1,C,2012-10-03,NA,0,2017/2018,139982,Academy converter,Grange Technology College,Secondary
966C673E1F91BCECFB3887D281FE0A39081A83B07359403F7833C67EF7818557,10875,2014/2015,1,1,1,C,2014-11-10,NA,0,2017/2018,135865,Academy sponsor led,Appleton Academy,Not applicable
56F243860E60E36A69DA5A81850F00AF5B84D5F9B60B6B23DA21828857B53AFB,9574,2015/2016,0,NA,NA,M,2012-10-04,2015-11-06,0,2017/2018,135367,Academy sponsor led,Bradford Academy,Not applicable
18746FF26D6AC95D757A4B4CB10B01DE6F041115B43ECC8BE0C4A0AEF575224B,14276,2015/2016,1,1,1,C,2012-10-04,NA,0,2017/2018,137065,Academy sponsor led,Coop Academy Leeds,Secondary
16AB3AC8B08A141C58F505036D9FC8AC09647F5485772F0849FC93AB9E7885EF,18286,2017/2018,1,1,1,C,2017-09-01,NA,0,2018/2019,145173,Academy sponsor led,Appleton Academy,NA
BFA4C3193490CCDA1B5422595F9E974355F83B6973445546D10283E5A3EE1296,18286,2017/2018,1,1,0,C,2017-09-01,2018-06-29,0,2018/2019,145173,Academy sponsor led,Appleton Academy,NA


In [394]:
ever_AP <- enrol_final |>
    filter(InstitutionTypeDesc == 'AP/PRU') 

In [396]:
ever_AP_ids <- ever_AP |>
    select(person_id) |>
    distinct() |>
    mutate(ever_AP = TRUE)

In [397]:
ever_AP_ids

person_id,ever_AP
<chr>,<lgl>
BC99204E62FE3D7E8E4D41B6D223F300ED59636D3A2E19BE5DCBAB85A8C6CA68,TRUE
0F719CE1354E346BC9788D861696C41D3E12CD4E9B922C321905506B99B7F517,TRUE
B606CEC2C788D17D26AE9F5AD9343D6047CF3C2905F4DD428783C1906106A262,TRUE
F8C2444EF68664B92F9D89F8AC9DFB216B728E3FFAAC2409F420B4D7C62CD3E0,TRUE
197739B330DCFC024B66615A0BA3C4DF10389D593D2D669EF5AB3EDDDEB69339,TRUE
FC4A9A9ECC56374706EC415D1FA433DF7C815C0F5048A15BF7AF96DF0CEF3933,TRUE
1CC67F0522C00BB034876FCEBBF85235B31DF165136E7754A30589AAA6CE5B5D,TRUE
3144635DF44220C33C414A245F1CD5FF2D3D95A5F3BEC9D7B86DC63C4004CC42,TRUE
C3976E86704D3DAA8186799F9A646B9EEF86B49204CFE99B8E18C66C67CAF8D4,TRUE


In [398]:
write.csv(ever_AP_ids, "data/ever_AP_ids.csv", row.names = FALSE)